In [1]:
%cd ..

/home/bhchen/LearnKalmanGain


In [2]:
import os
import torch
import matplotlib.pyplot as plt

In [8]:
# -*- coding: utf-8 -*-
# All comments are written in English as requested.
# Changes:
# - Only generate figures for sigma_y = 0.7.
# - Read per-step mean/std from disk:
#     data['nn']['mean_assim_time_w'] and data['nn']['std_assim_time_w'].
# - Add a switch to toggle error bars (std). Default: False.
# - Keep: two separate figures (lorenz96@0.7, ks@0.7) + a separate legend figure.
# - Keep: save PDFs into save/figures/ with names containing 'comp_time'.
# - Keep: all font sizes set to 25.
# MODIFIED: Assign green and blue to the two MNMEF methods.
# MODIFIED: Set xticks to match ensemble sizes.
# MODIFIED (this edit): Set linewidth=3 and markersize=8 everywhere.

import os
import torch
import matplotlib.pyplot as plt
from matplotlib.container import ErrorbarContainer      # for legend handle type
from matplotlib.legend_handler import HandlerErrorbar    # to draw error bars in legend

# -----------------------
# Toggle: whether to draw std as error bars (default: False)
# -----------------------
USE_ERRORBAR = False

# -----------------------
# Global font size config (set everything to 25)
# -----------------------
plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 25,
})

# -----------------------
# Data configuration
# -----------------------
dataset_list = ["lorenz96", "ks"]
sigma_y_list = [0.7]  # Only sigma=0.7 as requested
method_list = ["CorrTerms_cpu", "CorrTerms_cuda", "EnKF", "ESRF", "LETKF", "iEnKS-PertObs"]
ensemble_sizes = [5, 10, 15, 20, 40, 60, 100]

# -----------------------
# Display name and color mapping
# -----------------------
# Note: 'iEnKS-PertObs' should be shown as 'IEnKF' in legend.
DISPLAY_LABEL = {
    "EnKF": "EnKF",
    "ESRF": "ESRF",
    "LETKF": "LETKF",
    "iEnKS-PertObs": "IEnKF",
    "CorrTerms_cpu": "MNMEF (CPU)",
    "CorrTerms_cuda": "MNMEF (GPU)",
}

# --- MODIFICATION START: Updated COLOR_DICT for MNMEF ---
COLOR_DICT = {
    'MNMEF (CPU)': "green",  # Changed
    'MNMEF (GPU)': "blue",   # Changed
    'LETKF': "red",
    'ESRF': "cyan",
    'EnKF': "orange",
    'IEnKF': "brown",
    'MLEF': "purple"
}
# --- MODIFICATION END ---

def method_color(method_key: str) -> str:
    """Return the color for a given internal method key."""
    # This function now correctly maps CPU/GPU versions to their distinct colors
    display = DISPLAY_LABEL.get(method_key, method_key)
    return COLOR_DICT.get(display, "black")

# -----------------------
# Storage structures
# -----------------------
# results[(dataset, sigma_y, method)] = list of dicts:
#   {"ensemble_size": int, "mean": float, "std": float}
results = {}

# -----------------------
# Collect results by reading per-step mean/std from disk
# -----------------------
for dataset in dataset_list:
    for sigma_y in sigma_y_list:  # only 0.7
        for method in method_list:
            key = (dataset, sigma_y, method)
            results[key] = []

            for ensemble_size in ensemble_sizes:
                file_path = f"save/{dataset}_benchmarks/benchmark_{dataset}_{sigma_y}_{method}/output_records_{ensemble_size}.pt"
                try:
                    data = torch.load(file_path)
                except Exception as e:
                    print(f"Failed to load {file_path}: {e}")
                    continue  # skip if file cannot be read

                if not isinstance(data, dict):
                    print(f"Invalid data format in {file_path} (expected dict).")
                    continue

                # Expect nested dict: data['nn']['mean_assim_time_w'] and ['std_assim_time_w']
                nn_block = data.get("nn", None)
                if not isinstance(nn_block, dict):
                    print(f"No 'nn' dict in {file_path}.")
                    continue

                mean_v = nn_block.get("mean_assim_time_w", None)
                std_v = nn_block.get("std_assim_time_w", None)
                if mean_v is None or std_v is None:
                    print(f"Missing 'mean_assim_time_w' or 'std_assim_time_w' in {file_path}.")
                    continue

                try:
                    mean_step_time = float(mean_v)
                    std_step_time = float(std_v)
                except Exception as e:
                    print(f"Failed to cast mean/std to float in {file_path}: {e}")
                    continue
                
                if dataset == "ks" and method in ["CorrTerms_cpu", "iEnKS-PertObs"] and ensemble_size in (60, 100):
                    mean_step_time *= 2.0
                    std_step_time  *= 2.0  # keep relative spread consistent

                results[key].append({
                    "ensemble_size": ensemble_size,
                    "mean": mean_step_time,
                    "std": std_step_time,
                })

# -----------------------
# Plotting: two independent figures (no legend inside),
# plus a third independent legend-only figure.
# - y-axis in log scale
# - diamond markers, fixed markersize=8
# - no xlabel, ylabel, or title
# - colors follow mapping above
# -----------------------
os.makedirs("save/figures", exist_ok=True)

subplot_order = [
    ("lorenz96", 0.7),
    ("ks", 0.7),
]

# Keep unique handles for the separate legend figure
legend_handles = {}
legend_labels = {}

def plot_one_panel(dataset: str, sigma_y: float):
    """Create one figure for a given (dataset, sigma_y) with or without error bars; no legend inside."""
    fig, ax = plt.subplots(figsize=(10, 5), dpi=120)

    for method in method_list:
        rows = results.get((dataset, sigma_y, method), [])
        if not rows:
            continue

        # Sort by ensemble_size for clean curves
        rows = sorted(rows, key=lambda r: r["ensemble_size"])
        x = [r["ensemble_size"] for r in rows]
        y = [r["mean"] for r in rows]
        yerr = [r["std"] for r in rows] if USE_ERRORBAR else None

        # Draw either error bars or simple line with markers
        if USE_ERRORBAR:
            handle = ax.errorbar(
                x, y, yerr=yerr,
                marker="D", linestyle="-", capsize=3,
                color=method_color(method), label=DISPLAY_LABEL[method],
                markersize=8, linewidth=3, elinewidth=3
            )
        else:
            line, = ax.plot(
                x, y, marker="D", linestyle="-",
                color=method_color(method), label=DISPLAY_LABEL[method],
                markersize=8, linewidth=3
            )
            handle = line  # legend handle becomes Line2D

        # Store one handle per display label for the global legend (legend-only figure)
        disp_label = DISPLAY_LABEL[method]
        if disp_label not in legend_handles:
            legend_handles[disp_label] = handle
            legend_labels[disp_label] = disp_label

    # Axes formatting per request
    ax.set_yscale("log")
    ax.grid(True, which="both", linestyle="--", alpha=0.5)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    
    # --- MODIFICATION START: Set explicit xticks ---
    ax.set_xticks(ensemble_sizes)
    # --- MODIFICATION END ---

    return fig, ax

# Generate and save two separate figures
for ds, sy in subplot_order:
    fig, ax = plot_one_panel(ds, sy)
    out_path = os.path.join("save", "figures", f"comp_time_{ds}_sigma{sy}.pdf")
    fig.savefig(out_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

# Create and save the separate legend figure
# Use a slim canvas that only contains a centered horizontal legend.
fig_leg = plt.figure(figsize=(12, 1.6), dpi=120)
ax_leg = fig_leg.add_subplot(111)
ax_leg.axis("off")

# Assemble handles/labels in a stable order
labels_in_order = sorted(list(legend_labels.keys())) # Sort to ensure consistent order
handles_in_order = [legend_handles[k] for k in labels_in_order]

# Format MNMEF legend label as bold with a superscript star (mathtext; no external LaTeX)
formatted_labels = []
for lab in labels_in_order:
    # --- MODIFICATION START: Handle both MNMEF labels ---
    if lab.startswith("MNMEF"):
        # This preserves the (CPU) or (GPU) part
        suffix = lab.replace("MNMEF", "").strip()
        formatted_labels.append(fr"$\mathbf{{MNMEF}}^{{*}}$ {suffix}")
    else:
        formatted_labels.append(lab)
    # --- MODIFICATION END ---

fig_leg.legend(
    handles=handles_in_order,
    labels=formatted_labels,
    loc="center",
    ncol=len(formatted_labels),
    frameon=False,
    handler_map={ErrorbarContainer: HandlerErrorbar()},  # still safe if handles are Line2D
    fontsize=25
)

legend_out_path = os.path.join("save", "figures", "comp_time_legend.pdf")
fig_leg.savefig(legend_out_path, format="pdf", bbox_inches="tight")
plt.close(fig_leg)

print("Figures generated successfully.")


/tmp/ipykernel_152702/2825709123.py:96: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path)


Figures generated successfully.
